In [26]:
import mysql.connector
from decorator import contextmanager

In [ ]:
# form connection with the database

@contextmanager
def get_db_cursor(commit=False):

    # connection connects python to mysql server
    # stays open until closed
    connection = mysql.connector.connect(
        host = 'localhost',
        user = 'root',
        password = 'root',
        database = 'expense_manager'
    )
    # check if connected
    if connection.is_connected():
        print("Connection Successful")
    else:
        print("Failed in connecting to database")
    
    # Creates a cursor object.
    # A cursor is used to execute SQL queries and fetch results from the MySQL database.
    # it basically sends sql queries to the database.
    # workes through the connection
    cursor = connection.cursor(dictionary=True)

    yield cursor
    
    if commit:
        # it tells the database to permanently save the changes you made.
        connection.commit()
        # without commit changes are temporary in staging stage    

    cursor.close()
    connection.close()


In [28]:
# execute is used to run sql query
def fetch_all_records():
    with get_db_cursor() as cursor:
        cursor.execute("SELECT * FROM expenses")
        expenses = cursor.fetchall()   # gives output in a tuple format
        
        # if need in dictionary then do connection.cursor(dictionary=True)
        for expense in expenses:
            print(expense)

In [29]:
def fetch_expenses_for_date(expense_date):
    with get_db_cursor() as cursor:
        cursor.execute("SELECT * FROM expenses WHERE expense_date = %s" , (expense_date,))
        expenses = cursor.fetchall()
        for expense in expenses:
            print(expense)

In [ ]:
def insert_expense(expense_date, amount, category, notes):
    with get_db_cursor(commit=True) as cursor:
        cursor.execute("INSERT INTO expenses (expense_date, amount, category, notes) VALUES (%s, %s, %s, %s)",
                       (expense_date, amount, category, notes))

In [39]:
def delete_expense_for_date(expense_date):
    with get_db_cursor(commit=True) as cursor:
        cursor.execute("DELETE FROM expenses WHERE expense_date = %s", (expense_date,))
    

In [40]:
if __name__ == "__main__":
    print("----------expense for 08/04------------")
    #insert_expense('2024-08-04','30000','sip','investment')
    #fetch_all_records()
    fetch_expenses_for_date("2024-08-04")
    print("----------delete expense for 08/04-------------")
    delete_expense_for_date('2024-08-04')
    print("--------------expense for 08/04------------")
    fetch_expenses_for_date("2024-08-04")

----------expense for 08/04------------
Connection Successful
{'id': 19, 'expense_date': datetime.date(2024, 8, 4), 'amount': 25.0, 'category': 'Food', 'notes': 'Lunch'}
{'id': 20, 'expense_date': datetime.date(2024, 8, 4), 'amount': 200.0, 'category': 'Shopping', 'notes': 'Home supplies'}
{'id': 21, 'expense_date': datetime.date(2024, 8, 4), 'amount': 10.0, 'category': 'Other', 'notes': 'Parking'}
{'id': 69, 'expense_date': datetime.date(2024, 8, 4), 'amount': 30000.0, 'category': 'sip', 'notes': 'investment'}
----------delete expense for 08/04-------------
Connection Successful
--------------expense for 08/04------------
Connection Successful


In [ ]:
# A Context manager helps reduce redundant code by automatically handling setup and cleanup tasks.
# here it automatically handles closing resources after the context runs out and then executing the closing methods after it yields the cursor. 